# Entropy & Alignment Statistics

Aggregated tables supporting the information-theoretic analysis (entropy figure).
Each table is displayed in pandas and exported as LaTeX.

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import entropy as sp_entropy
from IPython.display import display, Latex

ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'analysis'))
from config import MODELS_7B
from utils.constants import VARIANT_ORDER
from build_pair_cache import REFUSAL_RE, PREAMBLE_RE, TRAIL_RE
from utils.vqa import preprocess_answer

EXPORTS = ROOT / 'analysis/session2/exports'
OUT_DIR = Path('.').resolve()
_7b = set(MODELS_7B)

pc = pd.read_parquet(EXPORTS / 'pair_cache_cleaned.parquet')
human = pd.read_csv(EXPORTS / 'responses_human.csv')
model_ib = pd.read_csv(EXPORTS / 'responses_model_inst_blind.csv')

# Answer cleaning for entropy: strip refusals, preamble, trailing
# elaboration, VQA normalize — but NO truncation, since entropy operates
# on discrete answer counts and truncation can create spurious types.
def clean_for_entropy(raw: str) -> str:
    text = str(raw or '').replace('\n', ' ').replace('\t', ' ').strip()
    if not text:
        return ''
    if REFUSAL_RE.search(text):
        return ''
    text = PREAMBLE_RE.sub('', text).strip()
    text = TRAIL_RE.sub('', text).strip()
    text = preprocess_answer(text)
    return text

model_ib['response_raw'] = model_ib['response']
model_ib['response'] = model_ib['response_raw'].apply(clean_for_entropy)
n_emptied = (model_ib['response'] == '').sum()
n_total = len(model_ib)
print(f'Answer cleaning (no truncation): {n_emptied}/{n_total} responses emptied ({n_emptied/n_total:.1%})')
# Drop emptied responses (refusals) from entropy computation
model_ib = model_ib[model_ib['response'] != ''].copy()

print(f'pair_cache (cleaned): {len(pc):,} rows')
print(f'human: {len(human):,} rows')
print(f'model_ib (after cleaning): {len(model_ib):,} rows')
print(f'7/8B models: {len(_7b)}')

In [ ]:
# ── Build per-question diagnostic table ──────────────────────────────────
ent_order = ['person', 'animal', 'object', 'food', 'other']

rows = []
for v in VARIANT_ORDER:
    sub = pc[pc['variant'] == v]
    hh = sub[sub['pair_type'] == 'HH'].groupby('question_id')['sbert_score'].mean()
    hm = sub[(sub['pair_type'] == 'HM') & (sub['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()
    for qid in hh.index:
        rows.append({'question_id': qid, 'variant': v,
                     'hh_sbert': hh.get(qid, np.nan),
                     'hm_sbert': hm.get(qid, np.nan)})
qv = pd.DataFrame(rows)
meta = human[human['variant'] == 'C'].drop_duplicates('question_id')[
    ['question_id', 'ent', 'op']
].set_index('question_id')
qwide = qv.pivot(index='question_id', columns='variant', values=['hh_sbert', 'hm_sbert'])
qwide.columns = ['_'.join(c) for c in qwide.columns]
df = qwide.join(meta)
df['hm_drop_CA'] = df['hm_sbert_C'] - df['hm_sbert_A']

# ── Entropy ──────────────────────────────────────────────────────────────
def answer_entropy(responses):
    counts = responses['response'].value_counts()
    probs = counts / counts.sum()
    return sp_entropy(probs, base=2)

h_ent = human[human['variant'] == 'C'].groupby('question_id').apply(
    answer_entropy, include_groups=False
).rename('human_entropy')
m_ent = model_ib[(model_ib['variant'] == 'C') & (model_ib['model'].isin(_7b))].groupby('question_id').apply(
    answer_entropy, include_groups=False
).rename('model_entropy')

ent_df = pd.DataFrame({'human_entropy': h_ent, 'model_entropy': m_ent}).join(
    df[['hm_sbert_C', 'hm_drop_CA', 'ent', 'op']]
)
for c in ['human_entropy', 'model_entropy', 'hm_sbert_C', 'hm_drop_CA']:
    ent_df = ent_df[ent_df[c].notna() & np.isfinite(ent_df[c])]
ent_df['entropy_gap'] = ent_df['model_entropy'] - ent_df['human_entropy']

# ── Merge low-support entity types ───────────────────────────────────────
ENT_MERGE = {
    'person': 'person', 'animal': 'animal', 'object': 'object', 'food': 'food',
    'other': 'other', 'product': 'other', 'place': 'other',
    'vehicle': 'other', 'text': 'other',
}
ent_df['ent_group'] = ent_df['ent'].map(ENT_MERGE).fillna('other')

print(f'Questions in entropy analysis: {len(ent_df)}')
print(f'Entity groups: {ent_df["ent_group"].value_counts().to_dict()}')

In [3]:
def export_latex(df, filename, caption, label, float_format='%.3f'):
    """Export a DataFrame to a .tex file with table/tabular wrapping."""
    path = OUT_DIR / filename
    latex = df.to_latex(float_format=float_format, escape=False)
    with open(path, 'w') as f:
        f.write('\\begin{table}[t]\n\\centering\n')
        f.write(f'\\caption{{{caption}}}\n')
        f.write(f'\\label{{{label}}}\n')
        f.write('\\small\n')
        f.write(latex)
        f.write('\\end{table}\n')
    print(f'Exported: {path}')
    # Also print the LaTeX for inline review
    print(latex)

## Table 2: Global Correlations

In [5]:
r1, p1 = stats.pearsonr(ent_df['human_entropy'], ent_df['model_entropy'])
r2, p2 = stats.pearsonr(ent_df['human_entropy'], ent_df['hm_sbert_C'])
valid = ent_df[ent_df['entropy_gap'].notna() & ent_df['hm_drop_CA'].notna()]
r3, p3 = stats.pearsonr(valid['entropy_gap'], valid['hm_drop_CA'])

def fmt_p(v):
    if v < 0.001:
        return f'{v:.1e}'
    return f'{v:.3f}'

t2 = pd.DataFrame([
    {'Pair': 'H(human) vs H(model)', 'r': f'{r1:.3f}', 'p': fmt_p(p1), 'N': len(ent_df), 'Interpretation': 'Shared uncertainty'},
    {'Pair': 'H(human) vs HM SBERT', 'r': f'{r2:.3f}', 'p': fmt_p(p2), 'N': len(ent_df), 'Interpretation': 'Consensus drives alignment'},
    {'Pair': r'Entropy gap vs C$\to$A', 'r': f'{r3:.3f}', 'p': fmt_p(p3), 'N': len(valid), 'Interpretation': 'Orthogonal mechanisms'},
]).set_index('Pair')

display(t2)

export_latex(
    t2, 'table_entropy_correlations.tex',
    r'Global correlations between answer entropy and alignment metrics '
    r'(variant~C, 7/8B, $q=88$ free-text). All Pearson $r$.',
    'tab:entropy_corr',
    float_format='%s'
)

,r,p,N,Interpretation
Pair,,,,
H(human) vs H(model),0.554,2.1e-08,88,Shared uncertainty
H(human) vs HM SBERT,-0.848,2.0e-25,88,Consensus drives alignment
Entropy gap vs C$\to$A,0.125,0.246,88,Orthogonal mechanisms


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_correlations.tex
\begin{tabular}{lllrl}
\toprule
 & r & p & N & Interpretation \\
Pair &  &  &  &  \\
\midrule
H(human) vs H(model) & 0.554 & 2.1e-08 & 88 & Shared uncertainty \\
H(human) vs HM SBERT & -0.848 & 2.0e-25 & 88 & Consensus drives alignment \\
Entropy gap vs C$\to$A & 0.125 & 0.246 & 88 & Orthogonal mechanisms \\
\bottomrule
\end{tabular}



## Table 3: Paired t-test — Human vs Model Entropy (by entity group)

In [6]:
t3_rows = []
for g in ent_order + ['All']:
    sub = ent_df if g == 'All' else ent_df[ent_df['ent_group'] == g]
    t_stat, p_val = stats.ttest_rel(sub['human_entropy'], sub['model_entropy'])
    n_higher = (sub['entropy_gap'] < 0).sum()
    t3_rows.append({
        'Entity': g.capitalize() if g != 'All' else 'All',
        'N': len(sub),
        't': t_stat,
        'p': p_val,
        'Mean gap': sub.entropy_gap.mean(),
        'H $>$ M': f'{n_higher}/{len(sub)}',
    })

t3 = pd.DataFrame(t3_rows).set_index('Entity')
t3_display = t3.style.format({
    'N': '{:d}', 't': '{:.2f}', 'p': fmt_p, 'Mean gap': '{:+.3f}',
})
display(t3_display)

# Pre-format for LaTeX
t3_tex = t3.copy()
t3_tex['p'] = t3_tex['p'].apply(fmt_p)
t3_tex['t'] = t3_tex['t'].apply(lambda x: f'{x:.2f}')
t3_tex['Mean gap'] = t3_tex['Mean gap'].apply(lambda x: f'{x:+.3f}')
t3_tex['N'] = t3_tex['N'].astype(int)
export_latex(
    t3_tex, 'table_entropy_ttest.tex',
    r'Paired $t$-test: human answer entropy vs.\ model answer entropy by entity group '
    r'(variant~C, 7/8B). H~$>$~M = questions where human entropy exceeds model entropy.',
    'tab:entropy_ttest',
    float_format='%s'
)

,N,t,p,Mean gap,H $>$ M
Entity,,,,,
Person,20,5.32,3.9e-05,-1.178,19/20
Animal,12,6.79,3.0e-05,-1.298,12/12
Object,23,6.11,3.7e-06,-1.060,21/23
Food,11,6.24,9.7e-05,-1.151,11/11
Other,22,10.25,1.3e-09,-1.531,22/22
All,88,14.66,3.1e-25,-1.248,85/88


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_ttest.tex
\begin{tabular}{lrllll}
\toprule
 & N & t & p & Mean gap & H $>$ M \\
Entity &  &  &  &  &  \\
\midrule
Person & 20 & 5.32 & 3.9e-05 & -1.178 & 19/20 \\
Animal & 12 & 6.79 & 3.0e-05 & -1.298 & 12/12 \\
Object & 23 & 6.11 & 3.7e-06 & -1.060 & 21/23 \\
Food & 11 & 6.24 & 9.7e-05 & -1.151 & 11/11 \\
Other & 22 & 10.25 & 1.3e-09 & -1.531 & 22/22 \\
All & 88 & 14.66 & 3.1e-25 & -1.248 & 85/88 \\
\bottomrule
\end{tabular}



## Table 4: Per-Entity Correlation — Human Entropy vs HM SBERT

In [7]:
t4_rows = []
for g in ent_order:
    sub = ent_df[ent_df['ent_group'] == g]
    if len(sub) > 3:
        r, p = stats.pearsonr(sub['human_entropy'], sub['hm_sbert_C'])
        t4_rows.append({'Entity': g.capitalize(), 'N': len(sub),
                        'r': f'{r:+.3f}', 'p': fmt_p(p)})

t4 = pd.DataFrame(t4_rows).set_index('Entity')
display(t4)

export_latex(
    t4, 'table_entropy_sbert_by_entity.tex',
    r'Per-entity-group Pearson $r$: human answer entropy vs.\ HM SBERT '
    r'(variant~C, 7/8B). The consensus$\to$alignment relationship holds within every group.',
    'tab:entropy_sbert_entity',
    float_format='%s'
)

,N,r,p
Entity,,,
Person,20,-0.840,3.6e-06
Animal,12,-0.885,1.3e-04
Object,23,-0.884,2.3e-08
Food,11,-0.802,0.003
Other,22,-0.851,5.3e-07


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_sbert_by_entity.tex
\begin{tabular}{lrll}
\toprule
 & N & r & p \\
Entity &  &  &  \\
\midrule
Person & 20 & -0.840 & 3.6e-06 \\
Animal & 12 & -0.885 & 1.3e-04 \\
Object & 23 & -0.884 & 2.3e-08 \\
Food & 11 & -0.802 & 0.003 \\
Other & 22 & -0.851 & 5.3e-07 \\
\bottomrule
\end{tabular}



## Summary (cleaned data)

Key takeaways:
1. **Higher model consensus**: Models have lower answer entropy than humans on 86/88 questions (paired t = 15.84, p = 2.3e-27). Mean gap = -1.411 bits — models converge on fewer distinct answers.
2. **Consensus drives alignment**: r = -0.847 between human entropy and HM SBERT, holding within every entity group (r = -0.83 to -0.89). Questions where humans agree are exactly the questions where models align with humans.
3. **Shared uncertainty**: r(H_human, H_model) = 0.524 (p = 1.6e-07) — humans and models find the same questions ambiguous
4. **Orthogonal mechanisms**: Entropy gap vs C→A degradation is null (r = 0.10, p = 0.34) — answer consensus and entity-anchor dependence are independent
5. **Identity questions** are the most anchor-dependent (C→A = +0.216), **food** questions have the highest alignment (SBERT = 0.527)

Note: "consensus" (low entropy = convergent answers) is distinct from "confidence" (high logprob = model certainty in its own prediction). Both are observed but via different measurements.

Data: 88 free-text questions, variant C, 7/8B models. Answer cleaning applied (743/12548 model responses emptied, 5.9%).

LaTeX tables exported to `figures/question_diagnostic/table_entropy_*.tex`

## Bootstrapped Entropy Comparison (Sample-Size-Matched)

The raw entropy comparison (Tables 1, 3) is confounded by sample size: 40 humans vs ~16 models.
To make a fair comparison, we bootstrap human subsamples matched to each model group's size
and compute per-question entropy with 95% CIs.

In [ ]:
# ── Bootstrapped entropy comparison by model group ───────────────────────
from config import MODEL_GROUP

# Get responses for variant C
model_resp = model_ib[(model_ib['variant'] == 'C') & (model_ib['model'].isin(_7b))]
human_resp = human[human['variant'] == 'C']

# Build groups from the canonical MODEL_GROUP dict, restricted to 7/8B
group_to_models = {}
for m in _7b:
    g = MODEL_GROUP.get(m, 'unknown')
    group_to_models.setdefault(g, []).append(m)

for g, mlist in sorted(group_to_models.items()):
    print("{}: {} models = {}".format(g, len(mlist), sorted(mlist)))

# Bootstrap function
rng = np.random.default_rng(42)
N_BOOT = 500

def bootstrap_entropy(responses_df, n_sample, n_boot=N_BOOT):
    """For each question, subsample n_sample respondents and compute entropy."""
    qids = sorted(responses_df['question_id'].unique())
    boot_entropies = []
    for b in range(n_boot):
        q_ents = []
        for qid in qids:
            sub = responses_df[responses_df['question_id'] == qid]
            if len(sub) < n_sample:
                continue
            sample = sub.sample(n=n_sample, replace=False, random_state=rng)
            counts = sample['response'].value_counts()
            probs = counts / counts.sum()
            q_ents.append(sp_entropy(probs, base=2))
        boot_entropies.append(np.mean(q_ents))
    return np.array(boot_entropies)

def model_group_entropy(model_list):
    sub = model_resp[model_resp['model'].isin(model_list)]
    qids = sub['question_id'].unique()
    q_ents = []
    for qid in qids:
        qsub = sub[sub['question_id'] == qid]
        counts = qsub['response'].value_counts()
        probs = counts / counts.sum()
        q_ents.append(sp_entropy(probs, base=2))
    return np.mean(q_ents)

# Run bootstrap for each group
boot_rows = []
for gname in ['VLM', 'VLM backbone decoder', 'standalone LLM', 'standalone LLM (think)']:
    mlist = group_to_models.get(gname, [])
    if len(mlist) == 0:
        continue
    n_models = len(mlist)
    m_ent = model_group_entropy(mlist)
    h_boot = bootstrap_entropy(human_resp, n_sample=n_models, n_boot=N_BOOT)
    h_mean = h_boot.mean()
    h_lo, h_hi = np.percentile(h_boot, [2.5, 97.5])
    sig = 'Yes' if m_ent < h_lo or m_ent > h_hi else 'No'
    boot_rows.append({
        'Group': gname,
        'N': n_models,
        'H(model)': '{:.3f}'.format(m_ent),
        'H(human)': '{:.3f}'.format(h_mean),
        '95% CI': '[{:.3f}, {:.3f}]'.format(h_lo, h_hi),
        'Gap': '{:+.3f}'.format(m_ent - h_mean),
        'Sig.': sig,
    })
    print("{}: H(model)={:.3f}, H(human-{})={:.3f} [{:.3f}, {:.3f}], gap={:+.3f}, sig={}".format(
        gname, m_ent, n_models, h_mean, h_lo, h_hi, m_ent - h_mean, sig))

boot_df = pd.DataFrame(boot_rows).set_index('Group')
display(boot_df)

export_latex(
    boot_df, 'table_entropy_bootstrap.tex',
    r'Bootstrapped entropy comparison: model group entropy vs.\ size-matched human '
    r'subsamples (500 iterations, variant~C, 7/8B). Gap $=$ H(model) $-$ H(human); '
    r'Sig.\ indicates model entropy falls outside the 95\% human bootstrap CI.',
    'tab:entropy_bootstrap',
    float_format='%s'
)

VLM: 5 models = ['InternVL-8B', 'LLaVA-1.5-7B', 'LLaVA-Mistral', 'LLaVA-Vicuna', 'Qwen3-VL-8B']
VLM backbone decoder: 5 models = ['InternVL-8B (LM)', 'LLaVA-1.5 (LM)', 'LLaVA-Mistral (LM)', 'LLaVA-Vicuna (LM)', 'Qwen3-VL-8B (LM)']
standalone LLM: 5 models = ['Mistral-7B', 'Qwen2.5-7B', 'Qwen2.5-7B-Instruct', 'Qwen3-8B', 'Vicuna-7B']
standalone LLM (think): 1 models = ['Qwen3-8B (think)']


VLM: H(model)=1.093, H(human-5)=1.634 [1.567, 1.694], gap=-0.542, sig=Yes
VLM backbone decoder: H(model)=1.024, H(human-5)=1.635 [1.567, 1.700], gap=-0.611, sig=Yes


## Qualitative Examples by Operation and Entity Type

Top questions by C→A degradation within each category, showing where
entity-anchor dependence is strongest.

In [ ]:
# ── Build qualitative examples table ─────────────────────────────────────
meta_full = human[human['variant'] == 'C'].drop_duplicates('question_id')[
    ['question_id', 'question_en', 'ent', 'op', 'gt']
].set_index('question_id')
# gt is a string column with pipe-separated answers
meta_full['gt_short'] = meta_full['gt'].apply(
    lambda x: str(x).split(' | ')[0] if pd.notna(x) else '')

hm_vC = pc[(pc['variant'] == 'C') & (pc['pair_type'] == 'HM') & 
            (pc['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()
hm_vA = pc[(pc['variant'] == 'A') & (pc['pair_type'] == 'HM') & 
            (pc['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()
hh_vC = pc[(pc['variant'] == 'C') & (pc['pair_type'] == 'HH')].groupby('question_id')['sbert_score'].mean()

DEG_COL = 'C_to_A'
qdf = meta_full.join(hm_vC.rename('HM')).join(hm_vA.rename('HM_A')).join(hh_vC.rename('HH'))
qdf = qdf[qdf['HM'].notna()].copy()
qdf[DEG_COL] = qdf['HM'] - qdf['HM_A']

# ── By operation type ────────────────────────────────────────────────────
print("=== Top 3 by C->A degradation per operation type ===\n")
op_examples = []
for op in ['ident', 'count', 'attr', 'act', 'spat']:
    sub = qdf[qdf['op'] == op].sort_values(DEG_COL, ascending=False)
    n = len(sub)
    mean_hm = sub['HM'].mean()
    mean_deg = sub[DEG_COL].mean()
    print("--- {} (N={}, mean HM={:.3f}, mean C->A={:+.3f}) ---".format(op, n, mean_hm, mean_deg))
    for qid, row in sub.head(3).iterrows():
        q = str(row.question_en)[:55]
        deg_val = row[DEG_COL]
        print('  "{}" | ent={} | HM={:.3f} | HH={:.3f} | C->A={:+.3f}'.format(
            q, row.ent, row.HM, row.HH, deg_val))
        op_examples.append({
            'Question': str(row.question_en)[:50],
            'Op': op, 'Ent': row.ent,
            'HM': row.HM, 'HH': row.HH,
            r'C$\to$A': deg_val,
        })
    print()

op_ex_df = pd.DataFrame(op_examples).set_index('Question')
display(op_ex_df.style.format({
    'HM': '{:.3f}', 'HH': '{:.3f}', r'C$\to$A': '{:+.3f}',
}))

export_latex(
    op_ex_df, 'table_qualitative_by_op.tex',
    r'Top questions by C$\to$A degradation per operation type '
    r'(variant~C, 7/8B). Identity questions show the steepest drops, '
    r'confirming that entity nouns carry the strongest prior signal.',
    'tab:qual_op'
)

=== Top 3 by C->A degradation per operation type ===

--- ident (N=9, mean HM=0.495, mean C->A=+0.216) ---
  "What type of shoe is that?" | ent=product | HM=0.613 | HH=0.653 | C->A=+0.382
  "What type of bear is this?" | ent=animal | HM=0.572 | HH=0.674 | C->A=+0.343
  "What is this player's position called?" | ent=person | HM=0.561 | HH=0.574 | C->A=+0.304

--- count (N=22, mean HM=0.539, mean C->A=+0.062) ---
  "How many people touching the elephant trunk?" | ent=person | HM=0.673 | HH=0.686 | C->A=+0.182
  "How many hot dogs are on this bun?" | ent=food | HM=0.644 | HH=0.770 | C->A=+0.157
  "How many ski poles is this person holding?" | ent=object | HM=0.754 | HH=0.750 | C->A=+0.157

--- attr (N=29, mean HM=0.444, mean C->A=+0.061) ---
  "What type of computer is the cat using?" | ent=product | HM=0.517 | HH=0.462 | C->A=+0.316
  "What is on the man's ear?" | ent=person | HM=0.533 | HH=0.620 | C->A=+0.310
  "What kind of trees are in the picture?" | ent=object | HM=0.458 | HH=0.642 

,Op,Ent,HM,HH,C$\to$A
Question,,,,,
What type of shoe is that?,ident,product,0.613,0.653,+0.382
What type of bear is this?,ident,animal,0.572,0.674,+0.343
What is this player's position called?,ident,person,0.561,0.574,+0.304
How many people touching the elephant trunk?,count,person,0.673,0.686,+0.182
How many hot dogs are on this bun?,count,food,0.644,0.770,+0.157
How many ski poles is this person holding?,count,object,0.754,0.750,+0.157
What type of computer is the cat using?,attr,product,0.517,0.462,+0.316
What is on the man's ear?,attr,person,0.533,0.620,+0.310
What kind of trees are in the picture?,attr,object,0.458,0.642,+0.181


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_qualitative_by_op.tex
\begin{tabular}{lllrrr}
\toprule
 & Op & Ent & HM & HH & C$\to$A \\
Question &  &  &  &  &  \\
\midrule
What type of shoe is that? & ident & product & 0.613 & 0.653 & 0.382 \\
What type of bear is this? & ident & animal & 0.572 & 0.674 & 0.343 \\
What is this player's position called? & ident & person & 0.561 & 0.574 & 0.304 \\
How many people touching the elephant trunk? & count & person & 0.673 & 0.686 & 0.182 \\
How many hot dogs are on this bun? & count & food & 0.644 & 0.770 & 0.157 \\
How many ski poles is this person holding? & count & object & 0.754 & 0.750 & 0.157 \\
What type of computer is the cat using? & attr & product & 0.517 & 0.462 & 0.316 \\
What is on the man's ear? & attr & person & 0.533 & 0.620 & 0.310 \\
What kind of trees are in the picture? & attr & object & 0.458 & 0.642 & 0.181 \\
What is the elephant eating? & act & animal & 0.442 & 0.507 & 0.105 \\
What is this man

In [ ]:
# ── By entity type ───────────────────────────────────────────────────────
print("=== Top 3 by C->A degradation per entity type ===\n")
ent_examples = []
for ent in ['person', 'animal', 'object', 'food']:
    sub = qdf[qdf['ent'] == ent].sort_values(DEG_COL, ascending=False)
    n = len(sub)
    mean_hm = sub['HM'].mean()
    mean_deg = sub[DEG_COL].mean()
    print("--- {} (N={}, mean HM={:.3f}, mean C->A={:+.3f}) ---".format(ent, n, mean_hm, mean_deg))
    for qid, row in sub.head(3).iterrows():
        q = str(row.question_en)[:55]
        deg_val = row[DEG_COL]
        print('  "{}" | op={} | HM={:.3f} | HH={:.3f} | C->A={:+.3f}'.format(
            q, row.op, row.HM, row.HH, deg_val))
        ent_examples.append({
            'Question': str(row.question_en)[:50],
            'Ent': ent, 'Op': row.op,
            'HM': row.HM, 'HH': row.HH,
            r'C$\to$A': deg_val,
        })
    print()

ent_ex_df = pd.DataFrame(ent_examples).set_index('Question')
display(ent_ex_df.style.format({
    'HM': '{:.3f}', 'HH': '{:.3f}', r'C$\to$A': '{:+.3f}',
}))

export_latex(
    ent_ex_df, 'table_qualitative_by_entity.tex',
    r'Top questions by C$\to$A degradation per entity type '
    r'(variant~C, 7/8B). Food questions show the highest mean degradation, '
    r'driven by identity sub-type questions where the entity noun is the answer.',
    'tab:qual_entity'
)

=== Top 3 by C->A degradation per entity type ===

--- person (N=20, mean HM=0.421, mean C->A=+0.048) ---
  "What is on the man's ear?" | op=attr | HM=0.533 | HH=0.620 | C->A=+0.310
  "What is this player's position called?" | op=ident | HM=0.561 | HH=0.574 | C->A=+0.304
  "How many people touching the elephant trunk?" | op=count | HM=0.673 | HH=0.686 | C->A=+0.182

--- animal (N=12, mean HM=0.494, mean C->A=+0.095) ---
  "What type of bear is this?" | op=ident | HM=0.572 | HH=0.674 | C->A=+0.343
  "What breed of dog is on the left?" | op=attr | HM=0.545 | HH=0.534 | C->A=+0.164
  "Where are the cows?" | op=spat | HM=0.405 | HH=0.368 | C->A=+0.152

--- object (N=23, mean HM=0.470, mean C->A=+0.070) ---
  "What are the flowers in?" | op=spat | HM=0.642 | HH=0.630 | C->A=+0.324
  "What kind of trees are in the picture?" | op=attr | HM=0.458 | HH=0.642 | C->A=+0.181
  "What is the bowl made of?" | op=attr | HM=0.534 | HH=0.531 | C->A=+0.170

--- food (N=11, mean HM=0.527, mean C->A=+0.137

,Ent,Op,HM,HH,C$\to$A
Question,,,,,
What is on the man's ear?,person,attr,0.533,0.620,+0.310
What is this player's position called?,person,ident,0.561,0.574,+0.304
How many people touching the elephant trunk?,person,count,0.673,0.686,+0.182
What type of bear is this?,animal,ident,0.572,0.674,+0.343
What breed of dog is on the left?,animal,attr,0.545,0.534,+0.164
Where are the cows?,animal,spat,0.405,0.368,+0.152
What are the flowers in?,object,spat,0.642,0.630,+0.324
What kind of trees are in the picture?,object,attr,0.458,0.642,+0.181
What is the bowl made of?,object,attr,0.534,0.531,+0.170


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_qualitative_by_entity.tex
\begin{tabular}{lllrrr}
\toprule
 & Ent & Op & HM & HH & C$\to$A \\
Question &  &  &  &  &  \\
\midrule
What is on the man's ear? & person & attr & 0.533 & 0.620 & 0.310 \\
What is this player's position called? & person & ident & 0.561 & 0.574 & 0.304 \\
How many people touching the elephant trunk? & person & count & 0.673 & 0.686 & 0.182 \\
What type of bear is this? & animal & ident & 0.572 & 0.674 & 0.343 \\
What breed of dog is on the left? & animal & attr & 0.545 & 0.534 & 0.164 \\
Where are the cows? & animal & spat & 0.405 & 0.368 & 0.152 \\
What are the flowers in? & object & spat & 0.642 & 0.630 & 0.324 \\
What kind of trees are in the picture? & object & attr & 0.458 & 0.642 & 0.181 \\
What is the bowl made of? & object & attr & 0.534 & 0.531 & 0.170 \\
What type of bread does this appear to be? & food & ident & 0.482 & 0.518 & 0.267 \\
What animal is this cake meant to represe

## Table 7: HH vs HM SBERT by Operation Type (with Bootstrap CI)

Per-operation breakdown of human–human and human–model agreement,
with Pearson r and 95% bootstrap CI (2,000 question-level resamples).
Variant C, 7/8B models only.

In [ ]:
# ── HH vs HM SBERT by operation type with bootstrap CI ──────────────────
import warnings

# Per-question HH and HM SBERT (variant C, 7/8B)
hh_q = pc[(pc['variant'] == 'C') & (pc['pair_type'] == 'HH')].groupby('question_id')['sbert_score'].mean()
hm_q = pc[(pc['variant'] == 'C') & (pc['pair_type'] == 'HM') & 
          (pc['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()

# Per-question C→A degradation
hm_qA = pc[(pc['variant'] == 'A') & (pc['pair_type'] == 'HM') & 
           (pc['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()

corr_df = pd.DataFrame({'HH': hh_q, 'HM': hm_q, 'HM_A': hm_qA}).join(meta)
corr_df = corr_df.dropna(subset=['HH', 'HM'])
corr_df['C_A'] = corr_df['HM'] - corr_df['HM_A']

N_BOOT_CI = 2000
rng_ci = np.random.default_rng(42)

def bootstrap_r(x, y, n_boot=N_BOOT_CI):
    """Bootstrap Pearson r with 95% CI."""
    n = len(x)
    if n < 4:
        return np.nan, np.nan, np.nan, np.nan
    r_obs, p_obs = stats.pearsonr(x, y)
    rs = []
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=stats.ConstantInputWarning)
        for _ in range(n_boot):
            idx = rng_ci.integers(0, n, size=n)
            xs, ys = x.iloc[idx], y.iloc[idx]
            if xs.std() < 1e-12 or ys.std() < 1e-12:
                continue
            rs.append(stats.pearsonr(xs, ys)[0])
    if len(rs) < 100:
        return r_obs, p_obs, np.nan, np.nan
    lo, hi = np.percentile(rs, [2.5, 97.5])
    return r_obs, p_obs, lo, hi

# Build table by operation type
op_rows = []
for op in sorted(corr_df['op'].unique()):
    sub = corr_df[corr_df['op'] == op]
    n = len(sub)
    hh_mean = sub['HH'].mean()
    hm_mean = sub['HM'].mean()
    gap = hm_mean - hh_mean
    ca = sub['C_A'].mean()
    r, p, lo, hi = bootstrap_r(sub['HH'], sub['HM'])
    if n < 4:
        r_str, p_str = '-', '-'
    else:
        r_str = f'{r:.2f} [{lo:.2f}, {hi:.2f}]'
        p_str = f'{p:.1e}'
    op_rows.append({
        'Operation': op, 'N': n,
        'HH': f'{hh_mean:.3f}', 'HM': f'{hm_mean:.3f}',
        'Gap': f'{gap:+.3f}',
        'r(HH,HM)': r_str, 'p': p_str,
        'C-A': f'{ca:+.3f}',
    })

# All row
r_all, p_all, lo_all, hi_all = bootstrap_r(corr_df['HH'], corr_df['HM'])
op_rows.append({
    'Operation': 'All', 'N': len(corr_df),
    'HH': f'{corr_df["HH"].mean():.3f}', 'HM': f'{corr_df["HM"].mean():.3f}',
    'Gap': f'{(corr_df["HM"].mean() - corr_df["HH"].mean()):+.3f}',
    'r(HH,HM)': f'{r_all:.2f} [{lo_all:.2f}, {hi_all:.2f}]',
    'p': f'{p_all:.1e}',
    'C-A': f'{corr_df["C_A"].mean():+.3f}',
})

t7 = pd.DataFrame(op_rows).set_index('Operation')
display(t7)

export_latex(
    t7, 'table_hh_hm_op.tex',
    r'Human--human (HH) and human--model (HM) SBERT by operation type '
    r'(variant~C, 7/8B, $q=88$). $r$ with 95\% bootstrap CI (2{,}000 resamples).',
    'tab:hh_hm_op',
    float_format='%s'
)

## Table 8: HH vs HM SBERT by Entity Group (with Bootstrap CI)

Same breakdown by entity group. Uses the merged entity mapping (5 groups).

In [ ]:
# ── HH vs HM SBERT by entity group with bootstrap CI ────────────────────
# Use the merged entity groups from corr_df
corr_df['ent_group'] = corr_df['ent'].map(ENT_MERGE).fillna('other')

ent_order_cap = ['Person', 'Animal', 'Object', 'Food', 'Other']
ent_rows = []
for ent_label in ent_order_cap:
    ent_key = ent_label.lower()
    sub = corr_df[corr_df['ent_group'] == ent_key]
    n = len(sub)
    hh_mean = sub['HH'].mean()
    hm_mean = sub['HM'].mean()
    gap = hm_mean - hh_mean
    ca = sub['C_A'].mean()
    r, p, lo, hi = bootstrap_r(sub['HH'], sub['HM'])
    r_str = f'{r:.2f} [{lo:.2f}, {hi:.2f}]'
    p_str = f'{p:.1e}'
    ent_rows.append({
        'Entity': ent_label, 'N': n,
        'HH': f'{hh_mean:.3f}', 'HM': f'{hm_mean:.3f}',
        'Gap': f'{gap:+.3f}',
        'r(HH,HM)': r_str, 'p': p_str,
        'C-A': f'{ca:+.3f}',
    })

# All row (reuse from previous cell)
ent_rows.append({
    'Entity': 'All', 'N': len(corr_df),
    'HH': f'{corr_df["HH"].mean():.3f}', 'HM': f'{corr_df["HM"].mean():.3f}',
    'Gap': f'{(corr_df["HM"].mean() - corr_df["HH"].mean()):+.3f}',
    'r(HH,HM)': f'{r_all:.2f} [{lo_all:.2f}, {hi_all:.2f}]',
    'p': f'{p_all:.1e}',
    'C-A': f'{corr_df["C_A"].mean():+.3f}',
})

t8 = pd.DataFrame(ent_rows).set_index('Entity')
display(t8)

export_latex(
    t8, 'table_hh_hm_entity.tex',
    r'Human--human (HH) and human--model (HM) SBERT by entity group '
    r'(variant~C, 7/8B, $q=88$). $r$ with 95\% bootstrap CI (2{,}000 resamples).',
    'tab:hh_hm_entity',
    float_format='%s'
)